# AiStock V11.1 — 行业/风格/规模轮动引擎测试

测试内容:
1. StyleRotationEngine 独立单元测试 (模拟数据)
2. 配置加载验证 (codes.yaml / market_state.yaml)
3. 7分量模型集成验证
4. 轮动可视化测试

In [1]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
from datetime import datetime
print(f'Test time: {datetime.now()}')

Test time: 2026-05-27 13:46:16.762689


## 1. StyleRotationEngine 单元测试

In [2]:
from subsystems.market_state.core.style_rotation_engine import (
    StyleRotationEngine, StyleRotationSignal, IndustryRotationSignal
)

# ── Mock TDX Adapter ──
class MockTDXAdapter:
    def get_index_daily(self, code, market_type='index_sh', count=120):
        np.random.seed(hash(code) % 2**31)
        dates = pd.date_range(end=datetime.now(), periods=count, freq='B')
        base = 1000 + np.random.randn() * 200

        # Create trending data for different industries
        trend = np.random.randn() * 0.5  # random trend
        noise = np.random.randn(count) * base * 0.02
        close = base + np.cumsum(noise) + np.arange(count) * trend
        close = np.maximum(close, 100)  # prevent negative

        return pd.DataFrame({
            'date': dates,
            'open': close * (1 + np.random.randn(count) * 0.01),
            'high': close * (1 + np.abs(np.random.randn(count) * 0.02)),
            'low': close * (1 - np.abs(np.random.randn(count) * 0.02)),
            'close': close,
            'volume': np.random.randint(100000, 1000000, count),
        })

    def get_bars(self, market_type, code, category=None, count=120):
        return self.get_index_daily(code, market_type, count)

    def get_macro_data(self, code, market=62, count=120):
        return self.get_index_daily(code, 'index_ext', count)

mock_tdx = MockTDXAdapter()
print('MockTDXAdapter created')

MockTDXAdapter created


In [4]:
# ── Test without ConfigService (using defaults) ──
engine = StyleRotationEngine(
    tdx_adapter=mock_tdx,
    config=None,
    cache=None,
)

print(f'Industry indices: {len(engine._industry_indices)}')
print(f'Style indices: {len(engine._style_indices)}')
print(f'Size indices: {len(engine._size_indices)}')
print(f'Weights: {engine._weights}')
# print(f'Alert thresholds: {engine._alert_thresholds}')

Industry indices: 10
Style indices: 4
Size indices: 6
Weights: {'industry': 0.35, 'style': 0.35, 'size': 0.3}


In [5]:
# ── Calculate Style Rotation Signal ──
signal = engine.calculate()

print(f'\n=== StyleRotationSignal ===')
print(f'Industry rotation count: {len(signal.industry_rotation)}')
print(f'Style signal: {signal.style_signal:.2f} [{signal.style_direction}]')
print(f'Size signal: {signal.size_signal:.2f} [{signal.size_direction}]')
print(f'Composite signal: {signal.composite_signal:.2f} [{signal.composite_direction}]')
print(f'Data available: {signal.data_available}')
print(f'Rotation alerts: {signal.rotation_alerts}')
print(f'\n=== Industry Heatmap Ranking ===')
for industry, value in signal.industry_heatmap_ranking:
    print(f'  {industry}: {value:.2f}')


=== StyleRotationSignal ===
Industry rotation count: 10
Style signal: -80.94 [value_leading]
Size signal: -41.52 [small_cap_leading]
Composite signal: -5.78 [neutral]
Data available: True
Rotation alerts: ['成长→价值风格切换 (5日比值变化: -13.4%)', '行业分化加剧 (领先:info_tech 14.7%, 滞后:finance -14.0%, 差值:28.7%)']

=== Industry Heatmap Ranking ===
  info_tech: 14.73
  telecom: 9.83
  healthcare: 9.59
  staples: 5.04
  energy: 0.75
  materials: 0.50
  industrials: -1.00
  discretionary: -1.22
  utilities: -9.56
  finance: -13.97


In [6]:
# ── Test to_dict serialization ──
signal_dict = signal.to_dict()
import json
print(json.dumps(signal_dict, indent=2, ensure_ascii=False)[:2000])

{
  "industry_rotation": {
    "finance": {
      "industry": "finance",
      "code": "932083",
      "name": "全指金融行业",
      "momentum_5d": 2.44,
      "momentum_20d": -13.97,
      "signal": 0.0,
      "direction": "bearish",
      "rank": 10
    },
    "energy": {
      "industry": "energy",
      "code": "932077",
      "name": "全指能源行业",
      "momentum_5d": 7.35,
      "momentum_20d": 0.75,
      "signal": 0.0,
      "direction": "neutral",
      "rank": 5
    },
    "materials": {
      "industry": "materials",
      "code": "932078",
      "name": "全指材料行业",
      "momentum_5d": 9.6,
      "momentum_20d": 0.5,
      "signal": 0.0,
      "direction": "neutral",
      "rank": 6
    },
    "industrials": {
      "industry": "industrials",
      "code": "932079",
      "name": "全指工业行业",
      "momentum_5d": -4.68,
      "momentum_20d": -1.0,
      "signal": 0.0,
      "direction": "neutral",
      "rank": 7
    },
    "discretionary": {
      "industry": "discretionary",
      "code

## 2. 配置加载验证

In [7]:
import yaml

# Load codes.yaml
with open('config/yaml/codes.yaml', 'r', encoding='utf-8') as f:
    codes_cfg = yaml.safe_load(f)

sr = codes_cfg['codes']['style_rotation']
print(f'Industry indices: {len(sr["industry_indices"])}')
print(f'Style indices: {len(sr["style_indices"])}')
print(f'Size indices: {len(sr["size_indices"])}')

# Verify all codes
for idx in sr['industry_indices']:
    print(f'  Industry: {idx["code"]} ({idx["name"]}) market={idx.get("market", "ST")}')

for idx in sr['style_indices']:
    print(f'  Style: {idx["code"]} ({idx["name"]}) role={idx["role"]}')

for idx in sr['size_indices']:
    print(f'  Size: {idx["code"]} ({idx["name"]}) tier={idx["tier"]}')

Industry indices: 10
Style indices: 4
Size indices: 6
  Industry: 932083 (全指金融行业) market=62
  Industry: 932077 (全指能源行业) market=62
  Industry: 932078 (全指材料行业) market=62
  Industry: 932079 (全指工业行业) market=62
  Industry: 932080 (全指可选行业) market=62
  Industry: 932081 (全指消费行业) market=62
  Industry: 932082 (全指医药行业) market=62
  Industry: 932084 (全指信息行业) market=62
  Industry: 932085 (全指通信行业) market=62
  Industry: 932086 (全指公用行业) market=62
  Style: 000918 (300成长) role=growth
  Style: 000919 (300价值) role=value
  Style: 399370 (国证成长) role=growth_alt
  Style: 399371 (国证价值) role=value_alt
  Size: 000016 (上证50) tier=large
  Size: 000300 (沪深300) tier=large
  Size: 000905 (中证500) tier=mid
  Size: 000852 (中证1000) tier=small
  Size: 932000 (中证2000) tier=micro
  Size: 399006 (创业板指) tier=gem


In [8]:
# Load market_state.yaml
with open('config/yaml/market_state.yaml', 'r', encoding='utf-8') as f:
    ms_cfg = yaml.safe_load(f)

ms = ms_cfg['market_state']
print(f'Version: {ms["version"]}')
print(f'\n7-component weights:')
weights = ms['composite_weights']
total = sum(weights.values())
for k, v in weights.items():
    print(f'  {k}: {v} (cumulative: {sum(list(weights.values())[:list(weights.keys()).index(k)+1]):.2f})')
print(f'Total weight: {total:.2f}')

print(f'\nStyle rotation weights: {ms["style_rotation_weights"]}')
print(f'Rotation alert thresholds: {ms["rotation_alert_thresholds"]}')

Version: 11.1

7-component weights:
  commodity: 0.2 (cumulative: 0.20)
  term_structure: 0.08 (cumulative: 0.28)
  index_basis: 0.15 (cumulative: 0.43)
  fund_flow: 0.15 (cumulative: 0.58)
  option_pcr: 0.12 (cumulative: 0.70)
  macro_valuation: 0.1 (cumulative: 0.80)
  style_rotation: 0.2 (cumulative: 1.00)
Total weight: 1.00

Style rotation weights: {'industry': 0.35, 'style': 0.35, 'size': 0.3}
Rotation alert thresholds: {'style_ratio_change_5d': 0.03, 'size_ratio_change_5d': 0.03, 'industry_spread': 15.0}


## 3. 7分量模型集成验证

In [9]:
from subsystems.market_state.core.derivatives_signal_engine import (
    DerivativesSignalEngine, DerivativesResult, _DEFAULT_COMPOSITE_WEIGHTS
)

# Verify 7-component default weights
print('Default composite weights (7-component):')
for k, v in _DEFAULT_COMPOSITE_WEIGHTS.items():
    print(f'  {k}: {v}')
print(f'Total: {sum(_DEFAULT_COMPOSITE_WEIGHTS.values()):.2f}')

# Verify DerivativesResult has style_rotation_signal field
result = DerivativesResult()
assert hasattr(result, 'style_rotation_signal'), 'Missing style_rotation_signal field'
print(f'\nDerivativesResult.style_rotation_signal: {result.style_rotation_signal}')
print('7-component model integration OK')

Default composite weights (7-component):
  commodity: 0.2
  term_structure: 0.08
  index_basis: 0.15
  fund_flow: 0.15
  option_pcr: 0.12
  macro_valuation: 0.1
  style_rotation: 0.2
Total: 1.00

DerivativesResult.style_rotation_signal: None
7-component model integration OK


## 4. 轮动可视化测试

In [10]:
from subsystems.market_state.visualization.plotly_visualizer import PlotlyVisualizer

viz = PlotlyVisualizer(output_dir='output/test_visualization')

# Generate rotation dashboard with mock data
signal_dict = signal.to_dict()
fig = viz.plot_style_rotation_dashboard(signal_dict)

# Save HTML
html_path = viz.save_html(fig, 'style_rotation_test')
print(f'Rotation dashboard saved: {html_path}')

PlotlyKeyError: Invalid property specified for object of type plotly.graph_objs.Indicator: 'xaxis'

Did you mean "align"?

    Valid properties:
        align
            Sets the horizontal alignment of the `text` within the
            box. Note that this attribute has no effect if an
            angular gauge is displayed: in this case, it is always
            centered
        customdata
            Assigns extra data each datum. This may be useful when
            listening to hover, click and selection events. Note
            that, "scatter" traces also appends customdata items in
            the markers DOM elements
        customdatasrc
            Sets the source reference on Chart Studio Cloud for
            `customdata`.
        delta
            :class:`plotly.graph_objects.indicator.Delta` instance
            or dict with compatible properties
        domain
            :class:`plotly.graph_objects.indicator.Domain` instance
            or dict with compatible propertie

In [ ]:
print('\n=== All V11.1 Style Rotation Tests PASSED ===')